<a href="https://colab.research.google.com/github/dhavanavijaywork/GenAI_Dhavana/blob/main/VectorDB_SemanticSearch_T5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q  -U sentence-transformers faiss-cpu chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import chromadb
print("Libraries imported successfully")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully")
documents = [
    "The minimum attendance requirement for students is 85 percent.",
    "Students must pay examination fees before the announced deadline.",
    "The college library closes at 8 PM on working days.",
    "Internal assessment marks are published on the student portal.",
    "Students can contact the department office for timetable questions.",
    "The computer laboratory is available during scheduled practical sessions."
]

for i, document in enumerate(documents, start=1):
    print(f"{i}. {document}")

Libraries imported successfully


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully
1. The minimum attendance requirement for students is 85 percent.
2. Students must pay examination fees before the announced deadline.
3. The college library closes at 8 PM on working days.
4. Internal assessment marks are published on the student portal.
5. Students can contact the department office for timetable questions.
6. The computer laboratory is available during scheduled practical sessions.


In [7]:
query = "What is the required attendance percentage?"
query_embedding = embedding_model.encode(
[query],
convert_to_numpy = True
)
print("Query:",query)
print("Query vector shape:",query_embedding.shape)
print("Length of documents list:", len(documents))
print("First vector values of query_embedding:", query_embedding[0])

Query: What is the required attendance percentage?
Query vector shape: (1, 384)
Length of documents list: 6
First vector values of query_embedding: [ 1.04030423e-01  5.59515692e-03 -2.36940831e-02 -1.05586760e-02
 -5.15882336e-02  4.73293439e-02 -2.81744767e-02 -1.32335154e-02
 -3.17729078e-02  1.54387415e-03  4.85915355e-02 -8.21098536e-02
  1.61670707e-02  2.42270213e-02  2.03179438e-02 -5.67601994e-02
  9.62223783e-02 -1.04901768e-01 -6.59815315e-03 -6.79493621e-02
 -4.29341607e-02 -1.86229814e-02  3.50394212e-02  2.13404577e-02
  4.45576422e-02 -5.06549813e-02  5.05868234e-02 -5.98538973e-05
  8.76898703e-04  3.58594768e-02 -3.44250537e-02  2.96056084e-02
  1.08420111e-01  1.53607745e-02 -9.24315862e-03  8.14516470e-03
  5.87152019e-02 -6.94415867e-02 -5.25811762e-02  5.71903288e-02
 -2.24945601e-02 -1.61334444e-02 -1.72070526e-02  5.95137514e-02
  1.62676070e-02  1.88143528e-03 -5.99443391e-02  3.92403007e-02
  4.62722406e-02  7.43615255e-02  6.77885413e-02  5.51871397e-02
  5.146

In [8]:
document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)
print("Document embeddings shape:", document_embeddings.shape)

Document embeddings shape: (6, 384)


In [11]:
dimension = document_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(document_embeddings.astype("float32"))
print("FAISS Index created")
print("Vectors stored:", faiss_index.ntotal)

FAISS Index created
Vectors stored: 6


In [15]:
k = 2  # Number of nearest neighbors to retrieve
distances, indices = faiss_index.search(query_embedding.astype("float32"), k)

print("Query:",query)
for rank, (distance, index) in enumerate(zip(distances[0], indices[0]), start=1):
    print(f"\nRank {rank}")
    print("Distance:", distance)
    print("Document:", documents[index])

Query: What is the required attendance percentage?

Rank 1
Distance: 0.43628594
Document: The minimum attendance requirement for students is 85 percent.

Rank 2
Distance: 1.2464015
Document: Students can contact the department office for timetable questions.


In [16]:
test_queries = [
    "When does the library shut?",
    "How can I pay my exam fees?",
    "Where are internal marks displayed?",
    "Who should I ask about the timetable?",
    "When can I use the computer lab?"
]

In [18]:
for query in test_queries:
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy = True
    ).astype("float32")
    distances, indices = faiss_index.search(query_vector, 1)
    best_index = indices[0][0]

    print("QUERY:", query)
    print("RETRIEVED:", documents[best_index])
    print("DISTANCE:", distances[0][0])
    print("-"*70)

QUERY: When does the library shut?
RETRIEVED: The college library closes at 8 PM on working days.
DISTANCE: 0.57711065
----------------------------------------------------------------------
QUERY: How can I pay my exam fees?
RETRIEVED: Students must pay examination fees before the announced deadline.
DISTANCE: 0.6211654
----------------------------------------------------------------------
QUERY: Where are internal marks displayed?
RETRIEVED: Internal assessment marks are published on the student portal.
DISTANCE: 0.8032142
----------------------------------------------------------------------
QUERY: Who should I ask about the timetable?
RETRIEVED: Students can contact the department office for timetable questions.
DISTANCE: 0.69258153
----------------------------------------------------------------------
QUERY: When can I use the computer lab?
RETRIEVED: The computer laboratory is available during scheduled practical sessions.
DISTANCE: 0.5136347
--------------------------------------

In [19]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="college_docs"
)
print("ChromaDB collection created")

ChromaDB collection created


In [23]:
document_ids = [f"doc_{i}" for i in range(len(documents))]

collection.add(
    ids=document_ids,
    documents=documents,
    embeddings=document_embeddings.tolist()
)
print("Documents added to ChromaDB collection.")
print("Stored records:", collection.count())

Documents added to ChromaDB collection.
Stored records: 6


In [24]:
query = "What is the required attendance percentage?"
query_embedding = embedding_model.encode(
[query],
convert_to_numpy = True
)[0].tolist()
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)
print("Query:", query)
for rank, document in enumerate(results["documents"][0], start=1):
    print(f"\nRank {rank}")
    print("Document:", document)

Query: What is the required attendance percentage?

Rank 1
Document: The minimum attendance requirement for students is 85 percent.

Rank 2
Document: Students can contact the department office for timetable questions.


In [25]:
metadata_collection = chroma_client.get_or_create_collection(
    name="college_docs_with_metadata"
)

metadata_values = [
    {"category": "attendance"},
    {"category": "fees"},
    {"category": "library"},
    {"category": "assessment"},
    {"category": "administration"},
    {"category": "laboratory"}
]

metadata_collection.add(
    ids=document_ids,
    documents=documents,
    embeddings=document_embeddings.tolist(),
    metadatas=metadata_values
)

print("Documents with metadata added.")

Documents with metadata added.


In [26]:
attendance_query = "How many classes must students attend?"

attendance_vector = embedding_model.encode(
    [attendance_query],
    convert_to_numpy=True
)[0].tolist()

filtered_results = metadata_collection.query(
    query_embeddings=[attendance_vector],
    n_results=2,
    where={"category": "attendance"}
)

print("Query:", attendance_query)

for document in filtered_results["documents"][0]:
    print(document)

Query: How many classes must students attend?
The minimum attendance requirement for students is 85 percent.


In [27]:
def semantic_search_faiss(query, top_k=3):
    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for distance, index in zip(distances[0], indices[0]):
        results.append({
            "document": documents[index],
            "distance": float(distance),
            "index": int(index)
        })

    return results


results = semantic_search_faiss(
    "Where can I find my internal marks?",
    top_k=3
)

for item in results:
    print(item)

{'document': 'Internal assessment marks are published on the student portal.', 'distance': 0.8999649286270142, 'index': 3}
{'document': 'Students can contact the department office for timetable questions.', 'distance': 1.7210479974746704, 'index': 4}
{'document': 'Students must pay examination fees before the announced deadline.', 'distance': 1.729201078414917, 'index': 1}
